# Preprocessing Pas à Pas — Irrigation Intelligente
### On va nettoyer et fusionner les 4 fichiers CSV ensemble

---
## ÉTAPE 1 — Importer les bibliothèques

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print('Bibliothèques importées ✓')

Bibliothèques importées ✓


---
## ÉTAPE 2 — Charger les 4 fichiers CSV

In [5]:
# Charger les 4 fichiers (ils sont dans le même dossier que ce notebook)
env   = pd.read_csv('stuard_environmental_data.csv')
soil  = pd.read_csv('stuard_soil_data.csv')
water = pd.read_csv('stuard_water_meter_data.csv')
ind   = pd.read_csv('indicators.csv')

print('Fichiers chargés ✓')
print(f'  Environmental : {env.shape}')
print(f'  Soil          : {soil.shape}')
print(f'  Water         : {water.shape}')
print(f'  Indicators    : {ind.shape}')

FileNotFoundError: [Errno 2] No such file or directory: 'stuard_environmental_data.csv'

---
## ÉTAPE 3 — Regarder ce qu'il y a dans chaque fichier

In [ ]:
# Regarder les 3 premières lignes de chaque fichier
print('=== ENVIRONMENTAL ===')
display(env.head(3))

print('=== SOIL ===')
display(soil.head(3))

print('=== WATER ===')
display(water.head(3))

print('=== INDICATORS ===')
display(ind.head(3))

---
## ÉTAPE 4 — Nettoyer le fichier ENVIRONMENTAL
### Problèmes à corriger :
- Le timestamp est en millisecondes → convertir en date lisible
- La colonne `battery` est presque vide → supprimer

In [ ]:
# ---- Voir les valeurs manquantes ----
print('Valeurs manquantes dans environmental :')
print(env.isnull().sum())

In [ ]:
# ---- Supprimer les colonnes inutiles ----
env = env.drop(columns=['battery', 'id', 'device_identifier'])

# ---- Convertir le timestamp ms en datetime ----
env['datetime'] = pd.to_datetime(env['ts_generation'], unit='ms', utc=True)
env = env.drop(columns=['ts_generation'])

# ---- Trier par date ----
env = env.sort_values('datetime').reset_index(drop=True)

print('Environmental nettoyé ✓')
display(env.head(3))
print('Colonnes :', list(env.columns))

---
## ÉTAPE 5 — Nettoyer le fichier SOIL
### Problèmes à corriger :
- Toutes les colonnes sont en format texte (string) → convertir en nombres
- 3 lignes de tomates (line 1, 2, 3) → on garde la colonne `line`

In [ ]:
print('Types actuels dans soil :')
print(soil.dtypes)

In [ ]:
# ---- Convertir les colonnes string en nombres ----
soil['ts_generation']           = pd.to_numeric(soil['ts_generation'],           errors='coerce')
soil['line']                    = pd.to_numeric(soil['line'],                    errors='coerce')
soil['electrical_conductivity'] = pd.to_numeric(soil['electrical_conductivity'], errors='coerce')
soil['humidity']                = pd.to_numeric(soil['humidity'],                errors='coerce')
soil['temperature']             = pd.to_numeric(soil['temperature'],             errors='coerce')

# ---- Supprimer les lignes avec NaN dans les colonnes importantes ----
soil = soil.dropna(subset=['ts_generation', 'line'])

# ---- Convertir le timestamp en datetime ----
soil['datetime'] = pd.to_datetime(soil['ts_generation'], unit='ms', utc=True)
soil['line']     = soil['line'].astype(int)

# ---- Supprimer les colonnes inutiles ----
soil = soil.drop(columns=['battery', 'id', 'device_identifier', 'ts_generation'])

# ---- Renommer pour éviter les confusions lors du merge ----
soil = soil.rename(columns={
    'humidity'              : 'soil_humidity',
    'temperature'           : 'soil_temperature',
    'electrical_conductivity': 'soil_ec'
})

soil = soil.sort_values(['line', 'datetime']).reset_index(drop=True)

print('Soil nettoyé ✓')
display(soil.head(3))
print('Lignes de tomates :', sorted(soil['line'].unique()))

---
## ÉTAPE 6 — Nettoyer le fichier WATER
### Problèmes à corriger :
- `current_volume` est **cumulatif** (compteur qui monte) → calculer la consommation réelle à chaque pas
- Exemple : si à 10h le compteur = 100L et à 10h10 il = 146L → consommation = 46L

In [ ]:
# Voir que le volume est bien cumulatif
print('Volume eau ligne 1 (premiers et derniers) :')
w1 = water[water['line'] == '1'].copy()
print('Min :', w1['current_volume'].astype(float).min())
print('Max :', w1['current_volume'].astype(float).max())
print('→ C\'est bien un compteur cumulatif qui monte de 0 à 38280 litres')

In [ ]:
# ---- Convertir les colonnes en nombres ----
water['ts_generation']  = pd.to_numeric(water['ts_generation'],  errors='coerce')
water['line']           = pd.to_numeric(water['line'],           errors='coerce')
water['current_volume'] = pd.to_numeric(water['current_volume'], errors='coerce')

water = water.dropna(subset=['ts_generation', 'line', 'current_volume'])

# ---- Convertir le timestamp ----
water['datetime'] = pd.to_datetime(water['ts_generation'], unit='ms', utc=True)
water['line']     = water['line'].astype(int)
water = water.drop(columns=['id', 'device_identifier', 'ts_generation'])
water = water.sort_values(['line', 'datetime']).reset_index(drop=True)

# ---- Calculer la consommation différentielle par ligne ----
# diff() calcule la différence entre chaque ligne et la ligne précédente
water['volume_L'] = water.groupby('line')['current_volume'].diff()

# Corriger les valeurs négatives (remise à zéro du compteur)
water.loc[water['volume_L'] < 0, 'volume_L'] = np.nan

# Remplir les NaN avec 0
water['volume_L'] = water['volume_L'].fillna(0)

# Supprimer la colonne cumulée
water = water.drop(columns=['current_volume'])

print('Water nettoyé ✓')
display(water.head(5))
print(f'Consommation moyenne par mesure (10 min) : {water["volume_L"].mean():.1f} L')

---
## ÉTAPE 7 — Nettoyer le fichier INDICATORS
### C'est le plus simple : données journalières, pas de problème majeur

In [ ]:
# Renommer la première colonne (timestamp)
ind = ind.rename(columns={'Unnamed: 0': 'ts_generation'})

# Convertir le timestamp en date (pas en datetime car c'est journalier)
ind['datetime'] = pd.to_datetime(ind['ts_generation'], unit='ms', utc=True)
ind['date']     = ind['datetime'].dt.date

ind = ind.drop(columns=['ts_generation', 'datetime'])
ind = ind.sort_values('date').reset_index(drop=True)

print('Indicators nettoyé ✓')
display(ind.head(3))

---
## ÉTAPE 8 — Rééchantillonner à 1 heure
### Pourquoi ? Les capteurs envoient des données toutes les 10 min.
### On regroupe par heure pour avoir un seul point par heure → plus facile à merger

In [ ]:
# ---- Environmental : arrondir à l'heure et faire la moyenne ----
env['datetime_h'] = env['datetime'].dt.floor('h')
env_h = env.groupby('datetime_h').mean(numeric_only=True).reset_index()
env_h = env_h.rename(columns={'datetime_h': 'datetime'})

print(f'Environmental : {len(env)} lignes (10 min) → {len(env_h)} lignes (1h)')
display(env_h.head(3))

In [ ]:
# ---- Soil : arrondir à l'heure, grouper par ligne ET heure ----
soil['datetime_h'] = soil['datetime'].dt.floor('h')
soil_h = soil.groupby(['line', 'datetime_h']).mean(numeric_only=True).reset_index()
soil_h = soil_h.rename(columns={'datetime_h': 'datetime'})

print(f'Soil : {len(soil)} lignes (10 min) → {len(soil_h)} lignes (1h)')
display(soil_h.head(3))

In [ ]:
# ---- Water : arrondir à l'heure, SOMMER (pas moyenne) le volume consommé ----
water['datetime_h'] = water['datetime'].dt.floor('h')
water_h = water.groupby(['line', 'datetime_h'])['volume_L'].sum().reset_index()
water_h = water_h.rename(columns={'datetime_h': 'datetime'})

print(f'Water : {len(water)} lignes (10 min) → {len(water_h)} lignes (1h)')
display(water_h.head(3))

---
## ÉTAPE 9 — MERGER les 4 fichiers
### Stratégie :
1. Pour chaque ligne de tomates (1, 2, 3) : joindre env + soil + water par datetime
2. Joindre les indicateurs par date
3. Mettre les 3 lignes ensemble

In [ ]:
# ---- Merger pour chaque ligne de tomates ----
toutes_les_lignes = []

for numero_ligne in [1, 2, 3]:
    print(f'Merge ligne {numero_ligne}...')
    
    # Prendre les données sol pour cette ligne
    soil_ligne  = soil_h[soil_h['line'] == numero_ligne].drop(columns='line')
    water_ligne = water_h[water_h['line'] == numero_ligne].drop(columns='line')
    
    # Merge 1 : environmental + soil (par datetime)
    df_ligne = pd.merge(env_h, soil_ligne, on='datetime', how='inner')
    
    # Merge 2 : + water (par datetime)
    df_ligne = pd.merge(df_ligne, water_ligne, on='datetime', how='inner')
    
    # Ajouter la date pour joindre les indicators
    df_ligne['date'] = df_ligne['datetime'].dt.date
    
    # Merge 3 : + indicators (par date)
    df_ligne = pd.merge(df_ligne, ind, on='date', how='left')
    
    # Garder le numéro de ligne
    df_ligne['line'] = numero_ligne
    
    toutes_les_lignes.append(df_ligne)
    print(f'  → {len(df_ligne)} lignes')

# Concatener les 3 lignes
df = pd.concat(toutes_les_lignes, ignore_index=True)
df = df.sort_values(['line', 'datetime']).reset_index(drop=True)

print(f'\nDataset final : {df.shape[0]} lignes × {df.shape[1]} colonnes')
display(df.head(5))

---
## ÉTAPE 10 — Vérifier le résultat

In [ ]:
print('Colonnes du dataset final :')
print(list(df.columns))

print('\nValeurs manquantes :')
print(df.isnull().sum())

print('\nTypes des colonnes :')
print(df.dtypes)

In [ ]:
print('Statistiques du dataset :')
display(df.describe().round(2))

---
## ÉTAPE 11 — Ajouter des features simples
### Ces features aident le modèle à mieux comprendre les données

In [ ]:
# ---- Feature 1 : Heure de la journée ----
df['heure'] = df['datetime'].dt.hour

# ---- Feature 2 : Encodage cyclique de l'heure ----
# Pourquoi ? Pour que le modèle sache que 23h et 0h sont proches
df['heure_sin'] = np.sin(2 * np.pi * df['heure'] / 24)
df['heure_cos'] = np.cos(2 * np.pi * df['heure'] / 24)

# ---- Feature 3 : VPD (Vapour Pressure Deficit) ----
# Mesure le stress hydrique de l'air. Plus VPD est élevé → plantes transpirent plus
T  = df['temperature']
RH = df['humidity']
es = 0.6108 * np.exp(17.27 * T / (T + 237.3))  # pression vapeur saturante (kPa)
df['vpd_kpa'] = es * (1 - RH / 100.0)
df['vpd_kpa'] = df['vpd_kpa'].clip(0, 10)

# ---- Feature 4 : Moyenne glissante 3h du volume d'eau ----
df['volume_L_roll_3h'] = (
    df.groupby('line')['volume_L']
    .transform(lambda x: x.rolling(3, min_periods=1).mean())
)

# ---- Feature 5 : Moyenne glissante 3h de l'humidité sol ----
df['soil_humidity_roll_3h'] = (
    df.groupby('line')['soil_humidity']
    .transform(lambda x: x.rolling(3, min_periods=1).mean())
)

print('Features ajoutées ✓')
print('Nouvelles colonnes :', ['heure', 'heure_sin', 'heure_cos', 'vpd_kpa',
                                'volume_L_roll_3h', 'soil_humidity_roll_3h'])

---
## ÉTAPE 12 — Créer la variable cible (TARGET)
### On veut prédire : combien d'eau sera consommée dans 1 heure ?

In [ ]:
# Décaler le volume d'une heure vers le futur (shift(-1))
df['target_volume_L'] = df.groupby('line')['volume_L'].shift(-1)

# Supprimer les dernières lignes sans cible
df = df.dropna(subset=['target_volume_L'])
df = df.reset_index(drop=True)

print(f'Dataset après création de la cible : {df.shape}')
print('\nDistribution de la cible :')
print(df['target_volume_L'].describe().round(2))

---
## ÉTAPE 13 — Visualiser les données

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# --- Température air ---
for ligne in [1, 2, 3]:
    d = df[df['line'] == ligne]
    axes[0,0].plot(d['datetime'], d['temperature'], label=f'Ligne {ligne}', alpha=0.7)
axes[0,0].set_title('Température air (°C)')
axes[0,0].legend()
axes[0,0].tick_params(axis='x', rotation=30)

# --- Humidité sol ---
for ligne in [1, 2, 3]:
    d = df[df['line'] == ligne]
    axes[0,1].plot(d['datetime'], d['soil_humidity'], label=f'Ligne {ligne}', alpha=0.7)
axes[0,1].set_title('Humidité sol (%)')
axes[0,1].legend()
axes[0,1].tick_params(axis='x', rotation=30)

# --- Volume eau consommé ---
for ligne in [1, 2, 3]:
    d = df[df['line'] == ligne]
    axes[1,0].plot(d['datetime'], d['volume_L'], label=f'Ligne {ligne}', alpha=0.7)
axes[1,0].set_title('Volume eau consommé (L/heure)')
axes[1,0].legend()
axes[1,0].tick_params(axis='x', rotation=30)

# --- GDD cumulé ---
axes[1,1].plot(df[df['line']==1]['datetime'], df[df['line']==1]['gdd'], color='green')
axes[1,1].set_title('GDD cumulé (Growing Degree Days)')
axes[1,1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('visualisation_donnees.png', dpi=100, bbox_inches='tight')
plt.show()
print('Graphique sauvegardé ✓')

---
## ÉTAPE 14 — Sauvegarder le dataset final

In [ ]:
# Supprimer les colonnes qu'on n'a plus besoin pour le modèle
df_final = df.drop(columns=['datetime', 'date', 'heure'])

# Sauvegarder
df_final.to_csv('dataset_final.csv', index=False)

print(f'Dataset sauvegardé : dataset_final.csv')
print(f'Shape : {df_final.shape[0]} lignes × {df_final.shape[1]} colonnes')
print(f'NaN : {df_final.isnull().sum().sum()}')
print('\nColonnes finales :')
for col in df_final.columns:
    print(f'  - {col}')

---
## ÉTAPE 15 — Split Train / Validation / Test
### IMPORTANT : On coupe dans le temps, pas aléatoirement !
### Si on mélange aléatoirement, le modèle verra des données futures → résultats faux

In [ ]:
n = len(df_final)

# 70% train / 15% validation / 15% test
train_end = int(n * 0.70)
val_end   = int(n * 0.85)

train = df_final.iloc[:train_end].copy()
val   = df_final.iloc[train_end:val_end].copy()
test  = df_final.iloc[val_end:].copy()

print(f'Train : {len(train)} lignes ({len(train)/n*100:.0f}%)')
print(f'Val   : {len(val)} lignes ({len(val)/n*100:.0f}%)')
print(f'Test  : {len(test)} lignes ({len(test)/n*100:.0f}%)')

# Sauvegarder les splits
train.to_csv('train.csv', index=False)
val.to_csv('val.csv',   index=False)
test.to_csv('test.csv',  index=False)

print('\nFichiers sauvegardés : train.csv, val.csv, test.csv ✓')

---
## ✅ RÉSUMÉ DE CE QU'ON A FAIT

| Étape | Action |
|-------|--------|
| 1-2 | Import bibliothèques + chargement CSV |
| 3 | Exploration des données |
| 4 | Nettoyage environmental (timestamp, battery) |
| 5 | Nettoyage soil (conversion string→number, renommage) |
| 6 | Nettoyage water (différentielle du volume cumulatif) |
| 7 | Nettoyage indicators (timestamp→date) |
| 8 | Rééchantillonnage 10min → 1h |
| 9 | **Merge des 4 sources** par datetime + date |
| 10 | Vérification du résultat |
| 11 | Features : heure sin/cos, VPD, rolling means |
| 12 | Cible : volume eau de la prochaine heure |
| 13 | Visualisation |
| 14 | Sauvegarde `dataset_final.csv` |
| 15 | Split temporel train/val/test |

### Fichiers générés :
- `dataset_final.csv` — dataset complet avec toutes les features
- `train.csv` — 70% des données pour entraîner le modèle
- `val.csv` — 15% pour ajuster les hyperparamètres
- `test.csv` — 15% pour évaluer le modèle final